# 04. Hybrid Recommendation

개인화 상품 추천 엔진 프로젝트의 Day 4 산출물입니다.  
이 노트북에서는 **Weighted Hybrid**와 **Switching Hybrid**를 비교하고, 정확도/커버리지/다양성 trade-off를 정리합니다.


## 체크리스트
- [x] Weighted Hybrid 구현 (CF score × α + CB score × β)
- [x] Switching Hybrid 구현 (sparse-profile user → CB, 그 외 → CF)
- [x] α, β grid search
- [ ] Feature Augmentation 검토
- [x] Hybrid vs baseline / CF / CB 비교표 작성
- [x] 정확도 vs 커버리지 trade-off 시각화
- [ ] Neural CF는 이번 범위에서 제외


In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), Path.cwd().parent] if (path / "src").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import FIGURES_DIR, HIGH_RATING_THRESHOLD, METRICS_DIR, RANDOM_SEED
from src.data import (
    build_relevance_sets,
    identify_cold_start_entities,
    load_bundle,
    random_train_test_split,
)
from src.evaluation import evaluate_recommendations
from src.features import (
    build_item_feature_matrix,
    build_item_similarity_matrix,
    build_tfidf_item_feature_matrix,
    build_user_profile_matrix,
    score_user_item_content,
)
from src.models import (
    blend_score_matrices,
    predict_user_based_scores,
    recommendation_cold_item_share,
    recommend_from_score_matrix,
    recommend_popular_items,
    switch_recommendations,
    user_activity_segments,
    users_below_interaction_threshold,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)


## 1. 데이터 로드 및 비교 기준
Day 1~3와 동일하게 **random seed=42** 기반 random split을 사용합니다.  
Hybrid의 diversity는 genre + decade 기반 item similarity로 계산하고, Content-Based 점수는 title+genre+decade TF-IDF metadata로 구성합니다.


In [2]:
bundle = load_bundle(download_if_missing=False)
ratings = bundle.ratings
items = bundle.items

train_df, test_df = random_train_test_split(ratings, test_size=0.2, random_state=RANDOM_SEED)
relevant_items = build_relevance_sets(test_df, min_rating=HIGH_RATING_THRESHOLD)
user_ids = sorted(relevant_items)
catalog = sorted(train_df["item_id"].unique())

base_item_features = build_item_feature_matrix(items)
item_similarity = build_item_similarity_matrix(base_item_features)
cold_start_entities = identify_cold_start_entities(train_df)

pd.Series(
    {
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "users_evaluated": len(user_ids),
        "cold_start_users_in_train": len(cold_start_entities["cold_start_users"]),
        "cold_start_items_in_train": len(cold_start_entities["cold_start_items"]),
    }
)


train_rows                   80000
test_rows                    20000
users_evaluated                926
cold_start_users_in_train        0
cold_start_items_in_train      355
dtype: int64

## 2. CF / Content-Based base score matrix 준비
- CF: Day 2에서 가장 강했던 **User-CF + Pearson + k=40**
- CB: MovieLens 100K의 tag 부재를 보완하기 위해 **title + genres + decade TF-IDF metadata**를 사용


In [3]:
popularity_recommendations = recommend_popular_items(
    train_df,
    user_ids,
    top_k=10,
    seen_interactions=train_df,
)

cf_score_matrix, _ = predict_user_based_scores(train_df, k=40, metric="pearson")
cf_recommendations = recommend_from_score_matrix(
    cf_score_matrix,
    train_df,
    user_ids=user_ids,
    top_k=10,
)
cf_topk_metrics = evaluate_recommendations(
    cf_recommendations,
    relevant_items,
    k=10,
    catalog=catalog,
)

tfidf_item_features = build_tfidf_item_feature_matrix(
    items,
    max_features=1500,
    min_df=2,
    ngram_range=(1, 2),
)
user_profiles = build_user_profile_matrix(
    train_df,
    tfidf_item_features,
    min_rating=HIGH_RATING_THRESHOLD,
)
content_score_matrix = score_user_item_content(user_profiles, tfidf_item_features)
content_recommendations = recommend_from_score_matrix(
    content_score_matrix,
    train_df,
    user_ids=user_ids,
    top_k=10,
)
content_topk_metrics = evaluate_recommendations(
    content_recommendations,
    relevant_items,
    k=10,
    catalog=catalog,
)

pd.DataFrame(
    [
        {"model_name": "cf_user_pearson_k40", **cf_topk_metrics},
        {"model_name": "content_tfidf_metadata", **content_topk_metrics},
    ]
)


,model_name,users_evaluated,precision@10,recall@10,ndcg@10,map@10,coverage
0,cf_user_pearson_k40,926.0,0.247948,0.278027,0.353363,0.226528,0.171913
1,content_tfidf_metadata,926.0,0.016631,0.020695,0.023895,0.010534,0.366223


## 3. Weighted Hybrid grid search
CF와 Content score를 **user-wise min-max normalization** 후 가중합합니다.  
여기서는 α ∈ {0.95, 0.90, 0.85, 0.80, 0.75, 0.70} 후보를 비교합니다.


In [4]:
def evaluate_topk_only(recommendations: dict[int, list[int]]) -> dict[str, float]:
    return evaluate_recommendations(
        recommendations,
        relevant_items,
        k=10,
        catalog=catalog,
    )

weighted_candidates: dict[float, dict[str, object]] = {}
weighted_rows: list[dict[str, float]] = []

for alpha in [0.95, 0.90, 0.85, 0.80, 0.75, 0.70]:
    score_matrix = blend_score_matrices(
        cf_score_matrix,
        content_score_matrix,
        alpha=alpha,
        normalization="user_minmax",
    )
    recommendations = recommend_from_score_matrix(
        score_matrix,
        train_df,
        user_ids=user_ids,
        top_k=10,
    )
    weighted_candidates[alpha] = {
        "score_matrix": score_matrix,
        "recommendations": recommendations,
    }
    weighted_rows.append(
        {
            "alpha": alpha,
            "beta": round(1.0 - alpha, 2),
            **evaluate_topk_only(recommendations),
        }
    )

weighted_grid_df = (
    pd.DataFrame(weighted_rows)
    .sort_values(["precision@10", "ndcg@10", "coverage"], ascending=[False, False, False])
    .reset_index(drop=True)
)
weighted_grid_df


,alpha,beta,users_evaluated,precision@10,recall@10,ndcg@10,map@10,coverage
0,0.85,0.15,926.0,0.249676,0.279275,0.357057,0.230615,0.180993
1,0.95,0.05,926.0,0.249460,0.280403,0.355838,0.228442,0.173123
2,0.90,0.10,926.0,0.248596,0.277927,0.356758,0.230221,0.181598
3,0.80,0.20,926.0,0.247840,0.277704,0.356324,0.229983,0.187046
4,0.75,0.25,926.0,0.245572,0.273947,0.351026,0.225377,0.197337
5,0.70,0.30,926.0,0.239957,0.267111,0.343884,0.219501,0.214286


In [5]:
best_weighted_row = weighted_grid_df.iloc[0]
best_weighted_alpha = float(best_weighted_row["alpha"])
best_weighted_score_matrix = weighted_candidates[best_weighted_alpha]["score_matrix"]
best_weighted_recommendations = weighted_candidates[best_weighted_alpha]["recommendations"]

best_weighted_row


alpha                0.850000
beta                 0.150000
users_evaluated    926.000000
precision@10         0.249676
recall@10            0.279275
ndcg@10              0.357057
map@10               0.230615
coverage             0.180993
Name: 0, dtype: float64

## 4. Switching Hybrid threshold search
strict cold-start(<5) 유저는 현재 random split 기준 거의 없으므로,  
실무적으로는 **interaction 수가 적은 sparse-profile user**를 대상으로 switching threshold를 탐색합니다.

선정 규칙:
1. CF 대비 Precision@10 감소폭이 **0.002 이하**인 후보만 남기고
2. 그 안에서 Coverage가 가장 높은 threshold를 선택합니다.


In [6]:
precision_tolerance = 0.002
switching_candidates: dict[int, dict[str, object]] = {}
switching_rows: list[dict[str, float]] = []

for threshold in [5, 10, 15, 20, 30, 40]:
    sparse_users = users_below_interaction_threshold(train_df, threshold=threshold)
    recommendations = switch_recommendations(
        cf_recommendations,
        content_recommendations,
        secondary_user_ids=sparse_users,
    )
    switching_candidates[threshold] = {
        "sparse_users": sparse_users,
        "recommendations": recommendations,
    }
    switching_rows.append(
        {
            "switch_threshold": threshold,
            "sparse_user_count": int(len(sparse_users)),
            **evaluate_topk_only(recommendations),
        }
    )

switching_grid_df = pd.DataFrame(switching_rows)
eligible_switching_df = switching_grid_df[
    switching_grid_df["precision@10"] >= (cf_topk_metrics["precision@10"] - precision_tolerance)
]
if eligible_switching_df.empty:
    best_switching_row = switching_grid_df.sort_values(
        ["precision@10", "coverage"], ascending=[False, False]
    ).iloc[0]
else:
    best_switching_row = eligible_switching_df.sort_values(
        ["coverage", "precision@10", "ndcg@10"], ascending=[False, False, False]
    ).iloc[0]

best_switching_threshold = int(best_switching_row["switch_threshold"])
best_switching_recommendations = switching_candidates[best_switching_threshold]["recommendations"]

switching_grid_df.sort_values("switch_threshold").reset_index(drop=True)


,switch_threshold,sparse_user_count,users_evaluated,precision@10,recall@10,ndcg@10,map@10,coverage
0,5,0,926.0,0.247948,0.278027,0.353363,0.226528,0.171913
1,10,1,926.0,0.247840,0.277667,0.353210,0.226488,0.176755
2,15,9,926.0,0.247192,0.276236,0.351804,0.225715,0.211259
3,20,113,926.0,0.235097,0.228414,0.318367,0.203186,0.354722
4,30,274,926.0,0.219978,0.180616,0.280798,0.178013,0.417676
5,40,376,926.0,0.203456,0.148486,0.249279,0.158343,0.433414


## 5. 최종 선택 모델 비교
선택 모델:
- popularity baseline
- best CF (user-pearson-k40)
- metadata TF-IDF content-based
- best weighted hybrid
- best switching hybrid


In [7]:
selected_models = {
    "popularity_baseline": popularity_recommendations,
    "cf_user_pearson_k40": cf_recommendations,
    "content_tfidf_metadata": content_recommendations,
    f"weighted_hybrid_alpha_{best_weighted_alpha:.2f}": best_weighted_recommendations,
    f"switching_hybrid_lt_{best_switching_threshold}": best_switching_recommendations,
}

comparison_rows: list[dict[str, float | str]] = []
for model_name, recommendations in selected_models.items():
    metrics = evaluate_recommendations(
        recommendations,
        relevant_items,
        k=10,
        catalog=catalog,
        item_similarity=item_similarity,
    )
    metrics["model_name"] = model_name
    metrics["cold_start_item_share"] = recommendation_cold_item_share(
        recommendations,
        cold_start_entities["cold_start_items"],
    )
    comparison_rows.append(metrics)

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values(["precision@10", "ndcg@10", "coverage"], ascending=[False, False, False])
    .reset_index(drop=True)
)
comparison_df


,users_evaluated,precision@10,recall@10,ndcg@10,map@10,intra_list_diversity,coverage,model_name,cold_start_item_share
0,926.0,0.249676,0.279275,0.357057,0.230615,0.594489,0.180993,weighted_hybrid_alpha_0.85,0.000000
1,926.0,0.247948,0.278027,0.353363,0.226528,0.620134,0.171913,cf_user_pearson_k40,0.000000
2,926.0,0.247192,0.276236,0.351804,0.225715,0.616017,0.211259,switching_hybrid_lt_15,0.001404
3,926.0,0.134665,0.140311,0.175515,0.090632,0.632658,0.029056,popularity_baseline,0.000000
4,926.0,0.016631,0.020695,0.023895,0.010534,0.094736,0.366223,content_tfidf_metadata,0.230022


## 6. 유저 활동량 segment별 비교
- cold_user: interaction < 5
- warm_user: 5 ≤ interaction < 20
- power_user: interaction ≥ 20

현재 split에서는 `cold_user`가 거의 없을 수 있으므로, warm/power 구간 비교가 더 중요합니다.


In [8]:
segments = user_activity_segments(train_df, cold_threshold=5, warm_threshold=20)
segment_rows: list[dict[str, float | str | int]] = []

for segment_name in ["cold_user", "warm_user", "power_user"]:
    segment_user_ids = segments[segments == segment_name].index
    segment_ground_truth = {
        user_id: relevant_items[user_id]
        for user_id in segment_user_ids
        if user_id in relevant_items
    }

    for model_name, recommendations in selected_models.items():
        scoped_recommendations = {
            user_id: recommendations[user_id]
            for user_id in segment_ground_truth
        }
        metrics = evaluate_recommendations(
            scoped_recommendations,
            segment_ground_truth,
            k=10,
            catalog=catalog,
        )
        metrics["segment"] = segment_name
        metrics["segment_users_in_train"] = int(len(segment_user_ids))
        metrics["model_name"] = model_name
        segment_rows.append(metrics)

segment_df = pd.DataFrame(segment_rows)
segment_df.sort_values(["segment", "precision@10"], ascending=[True, False]).head(15)


,users_evaluated,precision@10,recall@10,ndcg@10,map@10,coverage,segment,segment_users_in_train,model_name
0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,cold_user,0,popularity_baseline
1,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,cold_user,0,cf_user_pearson_k40
2,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,cold_user,0,content_tfidf_metadata
3,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,cold_user,0,weighted_hybrid_alpha_0.85
4,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,cold_user,0,switching_hybrid_lt_15
13,817.0,0.268421,0.260710,0.364179,0.234242,0.180387,power_user,830,weighted_hybrid_alpha_0.85
11,817.0,0.265973,0.257561,0.359795,0.229800,0.170702,power_user,830,cf_user_pearson_k40
14,817.0,0.265973,0.257561,0.359795,0.229800,0.170702,power_user,830,switching_hybrid_lt_15
10,817.0,0.145288,0.134551,0.184345,0.095541,0.029056,power_user,830,popularity_baseline
12,817.0,0.018360,0.022130,0.026036,0.011446,0.339588,power_user,830,content_tfidf_metadata


## 7. 아티팩트 저장
CSV/JSON/PNG를 `artifacts/metrics`, `artifacts/figures`에 저장합니다.


In [9]:
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

weighted_grid_path = METRICS_DIR / "day4_hybrid_grid_search.csv"
switching_grid_path = METRICS_DIR / "day4_switching_grid_search.csv"
comparison_path = METRICS_DIR / "day4_model_comparison.csv"
segment_path = METRICS_DIR / "day4_segment_comparison.csv"
summary_path = METRICS_DIR / "day4_hybrid_summary.json"

weighted_grid_df.sort_values("alpha", ascending=False).to_csv(weighted_grid_path, index=False)
switching_grid_df.sort_values("switch_threshold").to_csv(switching_grid_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
segment_df.to_csv(segment_path, index=False)

summary = {
    "best_weighted_alpha": best_weighted_alpha,
    "best_switching_threshold": best_switching_threshold,
    "switching_precision_tolerance": precision_tolerance,
    "cf_reference_metrics": cf_topk_metrics,
    "content_reference_metrics": content_topk_metrics,
    "selected_models": comparison_df.to_dict(orient="records"),
}
with summary_path.open("w", encoding="utf-8") as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
weighted_plot_df = weighted_grid_df.sort_values("alpha")
sns.lineplot(data=weighted_plot_df, x="alpha", y="precision@10", marker="o", ax=axes[0])
sns.lineplot(data=weighted_plot_df, x="alpha", y="coverage", marker="o", ax=axes[0])
axes[0].set_title("Weighted Hybrid Grid")
axes[0].set_ylabel("metric")

switching_plot_df = switching_grid_df.sort_values("switch_threshold")
sns.lineplot(data=switching_plot_df, x="switch_threshold", y="precision@10", marker="o", ax=axes[1])
sns.lineplot(data=switching_plot_df, x="switch_threshold", y="coverage", marker="o", ax=axes[1])
axes[1].set_title("Switching Threshold Search")
axes[1].set_ylabel("metric")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "day4_hybrid_grid.png", dpi=200, bbox_inches="tight")
plt.close(fig)

{
    "weighted_grid_path": str(weighted_grid_path.relative_to(PROJECT_ROOT)),
    "switching_grid_path": str(switching_grid_path.relative_to(PROJECT_ROOT)),
    "comparison_path": str(comparison_path.relative_to(PROJECT_ROOT)),
    "segment_path": str(segment_path.relative_to(PROJECT_ROOT)),
    "summary_path": str(summary_path.relative_to(PROJECT_ROOT)),
    "figure_path": str((FIGURES_DIR / "day4_hybrid_grid.png").relative_to(PROJECT_ROOT)),
}


{'weighted_grid_path': 'artifacts\\metrics\\day4_hybrid_grid_search.csv',
 'switching_grid_path': 'artifacts\\metrics\\day4_switching_grid_search.csv',
 'comparison_path': 'artifacts\\metrics\\day4_model_comparison.csv',
 'segment_path': 'artifacts\\metrics\\day4_segment_comparison.csv',
 'summary_path': 'artifacts\\metrics\\day4_hybrid_summary.json',
 'figure_path': 'artifacts\\figures\\day4_hybrid_grid.png'}

## 요약
- **Weighted Hybrid**는 CF 단독보다 Precision@10 / NDCG@10 / MAP을 소폭 개선하면서 Coverage도 함께 늘렸습니다.
- **Switching Hybrid**는 strict cold-start 유저가 거의 없는 split에서는 큰 정확도 향상은 없지만, sparse-profile user를 넓게 잡으면 Coverage를 늘리는 데 의미가 있습니다.
- 따라서 이번 실험에서는 **Weighted Hybrid를 기본 ranker**, **Switching Hybrid를 sparse-profile fallback 전략**으로 해석하는 것이 가장 자연스럽습니다.
